# PyDoseRT Optimized Figure (Axial + Sagittal + DVH)

This notebook wraps `print_paper_plot(...)` from `src/pydosert/utils/plotting.py` to generate the publication figure labeled:

- `PyDoseRT Optimized - axial`
- `PyDoseRT Optimized - sagittal`
- `Dose Volume Histogram (DVH)`

Data source:
- DICOM data from selected patient folder in `data/`
- predicted dose from `data/<patient>/pydosert_dose/dose_pred.npy`

## Review Notes (from `print_paper_plot`)

- The figure uses **predicted dose** for both axial/sagittal overlays and DVHs.
- DVH panel is not reference-vs-prediction; it is prediction-only, separated by structure.
- Isodose levels are plotted as filled bands with contour lines.
- Default dose scaling in this function uses `max(patient.dose, dose_pred)` in the loaded units.

In [ ]:
%matplotlib inline

from pathlib import Path
import sys
import numpy as np
import torch
from IPython.display import Image, display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repo root (missing pyproject.toml/src).")

repo_root = find_repo_root(Path.cwd())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from pydosert.data import OptimizationConfig, loaders
from pydosert.utils.utils import find_patient_paths
from pydosert.utils.plotting import print_paper_plot

print(f"Repo root: {repo_root}")

In [ ]:
# Optional helper: list available patient folders
data_root = repo_root / "data"
patients = sorted([p.name for p in data_root.iterdir() if p.is_dir()]) if data_root.exists() else []
print(f"Found {len(patients)} patient folders:")
print(patients)

In [ ]:
# -------------------------------
# User controls (edit this cell)
# -------------------------------
PATIENT_DIR = repo_root / "data" / "00LGN1CKZ"
PRED_DOSE_PATH = PATIENT_DIR / "pydosert_dose" / "dose_pred.npy"
OPTIMIZATION_JSON = repo_root / "src" / "pydosert" / "data" / "optimization_presets" / "vienna.json"

STRUCT_NAMES = ["CTVT", "PTV", "FemoralHead_L", "FemoralHead_R", "Bladder", "Rectum", "Body"]
NEW_SPACING = (3.0, 3.0, 3.0)
USE_DELIVERY = True
CROP_VOLUME = False

ISODOSE_PERCENT_LEVELS = (20, 40, 60, 80, 90, 95, 100, 105, 107, 110)
DOSE_ALPHA = 0.6
DOSE_CMAP = "turbo"

SAVE_FIGURE = True
SHOW_INLINE_FIGURE = True
OUT_DIR = repo_root / "out" / "publishing" / "optimized" / PATIENT_DIR.name
OUT_PATH = OUT_DIR / f"pydosert_optimized_{PATIENT_DIR.name}.png"

print(f"PATIENT_DIR: {PATIENT_DIR}")
print(f"PRED_DOSE_PATH: {PRED_DOSE_PATH}")
print(f"OUT_PATH: {OUT_PATH}")

In [ ]:
if not PATIENT_DIR.exists():
    raise FileNotFoundError(f"Patient directory not found: {PATIENT_DIR}")
if not PRED_DOSE_PATH.exists():
    raise FileNotFoundError(f"Predicted dose file not found: {PRED_DOSE_PATH}")

ct_folder, rtplan_path, rtdose_path, rtstruct_path = find_patient_paths(PATIENT_DIR)

patient, _ = loaders.load_dicom(
    ct_folder=ct_folder,
    dose_path=rtdose_path,
    plan_path=rtplan_path,
    struct_path=rtstruct_path,
    struct_names=STRUCT_NAMES,
    use_delivery=USE_DELIVERY,
    new_spacing=NEW_SPACING,
    crop_volume=CROP_VOLUME,
    device="cpu",
)

# Match the workflow used in test_vienna.py
if "Body" in patient.structures:
    patient.dose = patient.dose * patient.structures["Body"]

optimization = OptimizationConfig.from_json(OPTIMIZATION_JSON)

print("Loaded patient and optimization config.")
print(f"Dose shape: {tuple(patient.dose.shape)}")
print(f"Structures: {list(patient.structures.keys())}")
print(f"Fractions: {patient.number_of_fractions}")

In [ ]:
dose_pred_np = np.load(PRED_DOSE_PATH)
dose_pred = torch.from_numpy(dose_pred_np).to(dtype=torch.float32)

if dose_pred.ndim == 4:
    dose_pred = dose_pred[0]

if dose_pred.shape != patient.dose.shape:
    raise ValueError(
        f"Shape mismatch: dose_pred={tuple(dose_pred.shape)} vs patient.dose={tuple(patient.dose.shape)}"
    )

print(f"Prediction max (loaded units): {dose_pred.max().item():.3f}")
print(f"Reference max (loaded units): {patient.dose.max().item():.3f}")

In [ ]:
if SAVE_FIGURE:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

print_paper_plot(
    experiment=None,
    treatment=optimization,
    patient=patient,
    dose_pred=dose_pred,
    out_path=str(OUT_PATH) if SAVE_FIGURE else None,
    dose_alpha=DOSE_ALPHA,
    isodose_percent_levels=ISODOSE_PERCENT_LEVELS,
    cmap_dose=DOSE_CMAP,
)

if SAVE_FIGURE:
    print(f"Saved figure: {OUT_PATH}")
    if SHOW_INLINE_FIGURE:
        display(Image(filename=str(OUT_PATH)))
else:
    print("Displayed figure inline (not saved).")